# Machine Learning for Chemistry (CH4124)
## Assignment-Based Examination — Group B
Department of Chemistry, Indian Institute of Technology Guwahati

**Name:** Sristi Vats  **Roll No:** 230122057



---
## Question 1 — Reading a reaction SMILES


In [1]:
import numpy as np

reactions = [
    "CC(=O)O.OCC>[H+].[Cl-]>CC(=O)OCC.O",
    "C=C.[H][H]>[Pd]>CC",
    "c1ccccc1.O=[N+]([O-])O>>c1ccccc1[N+](=O)[O-].O",
]


In [2]:
def parse_reaction(rxn):
    fields = rxn.split(">")
    if len(fields) != 3:
        raise ValueError(f"expected exactly two '>' separators, found {len(fields) - 1}")
    reactants, reagents, products = fields

    def split_field(s):
        return [mol for mol in s.split(".") if mol != ""]

    return {
        "reactants": split_field(reactants),
        "reagents": split_field(reagents),
        "products": split_field(products),
    }

In [3]:
for i, rxn in enumerate(reactions, start=1):
    parsed = parse_reaction(rxn)
    print(f"Reaction {i}: {rxn}")
    for field_name, mols in parsed.items():
        print(f"  {field_name}: {len(mols)} molecule(s) -> {mols}")
    print()

Reaction 1: CC(=O)O.OCC>[H+].[Cl-]>CC(=O)OCC.O
  reactants: 2 molecule(s) -> ['CC(=O)O', 'OCC']
  reagents: 2 molecule(s) -> ['[H+]', '[Cl-]']
  products: 2 molecule(s) -> ['CC(=O)OCC', 'O']

Reaction 2: C=C.[H][H]>[Pd]>CC
  reactants: 2 molecule(s) -> ['C=C', '[H][H]']
  reagents: 1 molecule(s) -> ['[Pd]']
  products: 1 molecule(s) -> ['CC']

Reaction 3: c1ccccc1.O=[N+]([O-])O>>c1ccccc1[N+](=O)[O-].O
  reactants: 2 molecule(s) -> ['c1ccccc1', 'O=[N+]([O-])O']
  reagents: 0 molecule(s) -> []
  products: 2 molecule(s) -> ['c1ccccc1[N+](=O)[O-]', 'O']



**Answer**
Reaction 1 (acetic acid + ethanol -> ethyl acetate + water, catalysed by `[H+]`) is the **esterification**.
Reaction 3 has an empty reagent field (`>>`), meaning **no catalyst or reagent is specified** for that
transformation (the benzene nitration is written with reactants going directly to products).

---
## Question 2 — Balancing ethane combustion as a null-space problem

In [4]:
E = np.array([[2., 0., -1., 0.],
              [6., 0., 0., -2.],
              [0., 2., -2., -1.]])

In [5]:
U, S, Vt = np.linalg.svd(E)
rank = int(np.sum(S > 1e-10))
nullity = E.shape[1] - rank

x = Vt[-1]
x = x / x[0]

from fractions import Fraction
fracs = [Fraction(v).limit_denominator(1000) for v in x]
lcm = 1
for f in fracs:
    lcm = lcm * f.denominator // np.gcd(lcm, f.denominator)
x_whole = np.round(x * lcm).astype(int)

print("Singular values:", S)
print("Rank of E:", rank, " Nullity of E:", nullity)
print("Coefficients scaled to C2H6 = 1:", x)
print("Smallest whole-number coefficients (C2H6, O2, CO2, H2O):", x_whole)
print("Verification, E @ x (whole-number x):", E @ x_whole)

Singular values: [6.62561256 3.00720721 1.02857332]
Rank of E: 3  Nullity of E: 1
Coefficients scaled to C2H6 = 1: [1.  3.5 2.  3. ]
Smallest whole-number coefficients (C2H6, O2, CO2, H2O): [2 7 4 6]
Verification, E @ x (whole-number x): [0. 0. 0.]


**Answer**
The balanced equation is **2 C2H6 + 7 O2 -> 4 CO2 + 6 H2O**.
The null space of `E` has dimension one, which chemically means there is exactly **one
independent way to balance this reaction** (up to overall scale) — the atom-conservation
constraints pin down a unique stoichiometry, not a family of solutions.

---
## Question 3 — Hückel spectrum of butadiene .

In [6]:
A = np.zeros((4, 4))
for i in range(3):
    A[i, i + 1] = 1
    A[i + 1, i] = 1
print("Adjacency matrix A:\n", A)

Adjacency matrix A:
 [[0. 1. 0. 0.]
 [1. 0. 1. 0.]
 [0. 1. 0. 1.]
 [0. 0. 1. 0.]]


In [7]:
eigvals, eigvecs = np.linalg.eigh(A)
order = np.argsort(eigvals)[::-1]
eigvals_sorted = eigvals[order]

degree = A.sum(axis=1)

occ = np.array([2., 2., 0., 0.])
E_pi_beta_coeff = float(np.sum(occ * eigvals_sorted))
n_atoms = 4

deloc_beta = E_pi_beta_coeff - 4.0

print("Eigenvalues (descending, units of beta):", eigvals_sorted)
print("Degree of every atom:", degree)
print(f"E_pi = {n_atoms}*alpha + {E_pi_beta_coeff:.4f}*beta")
print(f"Delocalisation energy (relative to two isolated ethylenes) = {deloc_beta:.4f}*beta")

Eigenvalues (descending, units of beta): [ 1.61803399  0.61803399 -0.61803399 -1.61803399]
Degree of every atom: [1. 2. 2. 1.]
E_pi = 4*alpha + 4.4721*beta
Delocalisation energy (relative to two isolated ethylenes) = 0.4721*beta


**Answer**
Butadiene's delocalisation energy is about **0.472·β**, which is much smaller in magnitude
than benzene's **2·β**. This shows benzene's cyclic, fully conjugated π-system is far more
aromatically stabilised than butadiene's open (non-cyclic) conjugated chain.

---
## Question 4 — Oversmoothing on the benzene ring
.

In [8]:
A6 = np.zeros((6, 6))
for i in range(6):
    A6[i, (i + 1) % 6] = A6[(i + 1) % 6, i] = 1.0

H = np.random.default_rng(1).normal(size=(6, 3))

In [ ]:
A_tilde = A6 + np.eye(6)
D_tilde = np.diag(A_tilde.sum(axis=1))
P = np.linalg.inv(D_tilde) @ A_tilde
print("Row sums of P (should all be 1):", P.sum(axis=1))

Row sums of P (should all be 1): [1. 1. 1. 1. 1. 1.]


In [ ]:
eigvals_P = np.linalg.eigvals(P)
moduli = np.sort(np.abs(eigvals_P))[::-1]
mu = moduli[1]

print("Moduli of all six eigenvalues of P (descending):", moduli)
print(f"Smoothing rate mu (second-largest eigenvalue modulus) = {mu:.6f}")

Moduli of all six eigenvalues of P (descending): [1.00000000e+00 6.66666667e-01 6.66666667e-01 3.33333333e-01
 7.91848035e-17 3.92523115e-17]
Smoothing rate mu (second-largest eigenvalue modulus) = 0.666667


In [ ]:
feat = H.copy()
print("Largest deviation of any atom from the mean row, at each layer k:")
for k in range(0, 17):
    if k in (0, 1, 2, 4, 8, 16):
        mean_row = feat.mean(axis=0)
        dev = np.linalg.norm(feat - mean_row, axis=1)
        print(f"  k={k:2d}: max deviation from mean = {dev.max():.6f}")
    feat = P @ feat

Largest deviation of any atom from the mean row, at each layer k:
  k= 0: max deviation from mean = 1.241388
  k= 1: max deviation from mean = 0.536825
  k= 2: max deviation from mean = 0.345910
  k= 4: max deviation from mean = 0.154873
  k= 8: max deviation from mean = 0.030668
  k=16: max deviation from mean = 0.001197


**Answer**
With smoothing rate μ ~ 0.667<1, the deviation of any atom from the mean feature row decays
geometrically like μᵏ, dropping from ~1.24 at k=0 to ~0.001 by k=16. For a molecule of about a
dozen heavy atoms, this means **only a small number of message-passing layers should be used** before the propagated features become nearly indistinguishable across atoms
(oversmoothing-deeper GNNs need residual connections or fewer layers to remain useful.

---
## Question 5 — Principal component analysis of two descriptors

In [9]:
X = np.array([[ 78.1, 2.3],
              [ 92.1, 1.7],
              [106.2, 2.8],
              [120.2, 2.0],
              [134.2, 3.1],
              [148.2, 2.4]])

N = X.shape[0]
def fix_sign(v):
    v = v.copy()
    if v[np.argmax(np.abs(v))] < 0:
        v = -v
    return v

In [10]:
Xc = X - X.mean(axis=0)
cov_c = (Xc.T @ Xc) / N
eigvals_c, eigvecs_c = np.linalg.eigh(cov_c)
order_c = np.argsort(eigvals_c)[::-1]
eigvals_c = eigvals_c[order_c]
eigvecs_c = eigvecs_c[:, order_c]
frac_c = eigvals_c / eigvals_c.sum()
pc1_c = fix_sign(eigvecs_c[:, 0])

print("Centred PCA")
print("Covariance matrix:\n", cov_c)
print("Eigenvalues:", eigvals_c)
print("Fraction of variance explained:", frac_c)
print("First principal direction:", pc1_c)

Centred PCA
Covariance matrix:
 [[5.73535556e+02 4.56277778e+00]
 [4.56277778e+00 2.18055556e-01]]
Eigenvalues: [5.73571866e+02 1.81744746e-01]
Fraction of variance explained: [9.99683236e-01 3.16764448e-04]
First principal direction: [0.99996834 0.0079578 ]


In [11]:
Xs = Xc/Xc.std(axis=0)
cov_s = (Xs.T @ Xs) / N
eigvals_s, eigvecs_s = np.linalg.eigh(cov_s)
order_s = np.argsort(eigvals_s)[::-1]
eigvals_s = eigvals_s[order_s]
eigvecs_s = eigvecs_s[:, order_s]
frac_s = eigvals_s / eigvals_s.sum()
pc1_s = fix_sign(eigvecs_s[:, 0])

print("Standardised PCA")
print("Covariance matrix:\n", cov_s)
print("Eigenvalues:", eigvals_s)
print("Fraction of variance explained:", frac_s)
print("First principal direction:", pc1_s)

Standardised PCA
Covariance matrix:
 [[1.         0.40800508]
 [0.40800508 1.        ]]
Eigenvalues: [1.40800508 0.59199492]
Fraction of variance explained: [0.70400254 0.29599746]
First principal direction: [0.70710678 0.70710678]


**Answer**
The two analyses disagree because molar mass has a much larger numeric range/variance than
dipole moment, so centred PCA lets molar mass dominate the first principal component
(loading ~0.9999 on mass, essentially ignoring dipole moment); standardising puts both
descriptors on an equal footing (loadings ~0.707/0.707), giving a first component that
genuinely reflects both properties-**the standardised analysis is the one that should be
reported.**